In [ ]:
import ROOT, glob, os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

In [ ]:
dirDir= "/home/users/l/lebrun/"
resultPath = dirDir + "results/"
figPath = dirDir + "figures/"
dataPath = "/dpnc/beegfs/dampe/tryf/ForKimberly/"
exposurePath = "/dpnc/beegfs/dampe/tryf/ForKimberly/exposureMaps_GIE/"
exponent = 0


In [ ]:
from astropy.time import TimeDelta, Time

RA_target = 94.3
DEC_target = -7.8
target_name = "KM3-230213A"

r50 = 1.2
r68 = 1.5
r90 = 2.2
r99 = 3

# MJD_target = Time(59988.0533299, format='mjd')
MJD_target = Time(58892, format='mjd')

six_months = TimeDelta(182.625, format='jd')  # 6 months ≈ 182.625 days

start_date = MJD_target - six_months.jd
end_date = MJD_target + six_months.jd



In [ ]:
from astropy.time import Time, TimeDelta
import astropy.units as u
from astropy.coordinates import EarthLocation

LEAP_SECONDS = [
    "2015-07-01T00:00:00",  # +1 leap second
    "2017-01-01T00:00:00",  # +1 leap second
]

def add_leap_seconds(mjd1, mjd2):
    times = Time(mjd1, mjd2, scale='tt', format='mjd')

    total_correction = np.zeros(len(times))  # seconds

    for leap in LEAP_SECONDS:
        time_leap = Time(leap, scale='tt')
        total_correction += (times > time_leap).astype(int) # Adds 1 or 0 -> Plays on boolean type

    corrected_times = times + TimeDelta(total_correction, format='sec')

    mjd1_new = corrected_times.tt.jd1 - 2400000.5
    mjd2_new = corrected_times.tt.jd2

    return mjd1_new, mjd2_new

def met_to_mjd_tt_split(seconds, milliseconds):
    # Total delta since epoch: seconds + milliseconds, kept separate
    seconds = np.asarray(seconds)
    milliseconds = np.asarray(milliseconds)

    # Construct TimeDelta in two parts to preserve precision
    delta_sec = TimeDelta(seconds, format='sec')
    delta_ms  = TimeDelta(milliseconds * 1e-3, format='sec')  # convert ms to s

    # MET epoch in TT 
    epoch = Time("2013-01-01T00:00:00", scale="utc")

    full_time = epoch + delta_sec + delta_ms

    # Get split MJD
    mjd1 = full_time.tt.jd1 - 2400000.5
    mjd2 = full_time.tt.jd2

    return mjd1, mjd2

def ecef_to_eci(mjd1, mjd2, x, y, z):
    t = Time(mjd1, mjd2, scale='tt', format='mjd')

    if isinstance(x, pd.Series):
        x = x.to_numpy() * u.m
    if isinstance(y, pd.Series):
        y = y.to_numpy() * u.m
    if isinstance(z, pd.Series):
        z = z.to_numpy() * u.m

    loc_ecef = EarthLocation(x=x, y=y, z=z)

    pos_gcrs, vel_gcrs = loc_ecef.get_gcrs_posvel(t)


    # TODO when we have sat velocities, figure out whether we can correct them too
    return pos_gcrs.get_xyz()




In [ ]:
dataDir = "/home/users/l/lebrun/data/"
dataName = "DmlNtup_Data_9years_FullPhotonData_AllSky.root"
tree = None
f = ROOT.TFile(dataDir + dataName, "READ") 

tree = f.Get("DmlNtup")
print(tree)
print(type(tree))
rdf = ROOT.RDataFrame(tree)

rdf = rdf.Filter(f"(celestialcoordBest_0 > {np.deg2rad(RA_target - r99)}) && (celestialcoordBest_0 < {np.deg2rad(RA_target + r99)})"
                  ).Filter(f"(celestialcoordBest_1 > {np.deg2rad(DEC_target - r99)}) && (celestialcoordBest_1 < {np.deg2rad(DEC_target + r99)})")

df = rdf.AsNumpy(["celestialcoordBest_0", "celestialcoordBest_1", "tt_bgoTotalEcorr_GeV", 'tt_second', 'tt_milisecond'])
df = pd.DataFrame(df)

df = df.rename(columns={"celestialcoordBest_0" : "RA", "celestialcoordBest_1" : "DEC", 
                        "tt_bgoTotalEcorr_GeV" : "energy"})

In [ ]:
print(len(df))
print(f"RA spans from {df.RA.min()} rad to {df.RA.max()} rad")
print(f"RA spans from {np.rad2deg(df.RA.min())} deg to {np.rad2deg(df.RA.max())} deg")
print(f"RA spans from {df.DEC.min()} rad to {df.DEC.max()} rad")
print(f"RA spans from {np.rad2deg(df.DEC.min())} deg to {np.rad2deg(df.DEC.max())} deg")

In [ ]:
df["MJD1"], df["MJD2"] = met_to_mjd_tt_split(df["tt_second"], df["tt_milisecond"])
df["MJD1"], df["MJD2"] = add_leap_seconds(df["MJD1"].values, df["MJD2"].values)

# df['X_ECI'], df['Y_ECI'], df['Z_ECI'] = ecef_to_eci(df["MJD1"], df["MJD2"],
#                                                      df['tt_sat_position_x'],
#                                                      df['tt_sat_position_y'],
#                                                      df['tt_sat_position_z'])

print(f"Minimal day : {df['MJD1'].min()}, max day : {df['MJD1'].max()}")

In [ ]:
# Filter df to only include events within ± six months of MJD_target
event_times = Time(df["MJD1"].values, df["MJD2"].values, format="mjd", scale="tt")
mask = (event_times >= start_date) & (event_times <= end_date)
df_six_months = df[mask]
print(f"Number of events within ± six months: {len(df_six_months)}")

In [ ]:
# Convert RA/DEC from radians to degrees for plotting
ra_deg = np.rad2deg(df_six_months['RA'])
dec_deg = np.rad2deg(df_six_months['DEC'])

# Define bin edges
ra_min = RA_target - r99
ra_max = RA_target + r99
dec_min = DEC_target - r99
dec_max = DEC_target + r99
fig, ax = plt.subplots(figsize=(6, 5))

# Plot 2D histogram
h = ax.hist2d(
    ra_deg, dec_deg,
    bins=50,
    range=[[ra_min, ra_max], [dec_min, dec_max]],
    cmap='viridis'
)

plt.colorbar(h[3], ax=ax, label='Counts')

# Draw a star at (RA_target, DEC_target)
star_handle, = ax.plot(RA_target, DEC_target, marker='*', linestyle='None', color='red', markersize=12, label=f'{target_name}')

# Draw circles at r50, r68, r90, r99 with different linestyles, all red
linestyles = ['dotted', 'dashed', 'dashdot', 'solid']
circle_handles = []
for r, ls, label in zip([ r68, r90, r99], linestyles, [ 'r68', 'r90', 'r99']):
    circle = plt.Circle((RA_target, DEC_target), r, color='r', fill=False, lw=1.5, linestyle=ls, label=label)
    ax.add_patch(circle)
    circle_handles.append(circle)

# Add legend for circles and target above the plot, horizontally
handles = [star_handle] + circle_handles 
labels = [target_name, 'R(68%)', 'R(90%)', 'R(99%)']
ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=5, frameon=False)

ax.set_xlabel('RA [deg]')
ax.set_ylabel('DEC [deg]')
str_start = start_date.strftime('%Y-%m-%d')
str_end = end_date.strftime('%Y-%m-%d')
ax.set_title(f'Events from {str_start} to {str_end}')
plt.tight_layout()
fig.savefig(figPath+f'explo/{target_name}.png')
plt.show()


In [ ]:
HKdataPath = "/dpnc/beegfs/dampe/prod/FM/skim/6.0.0/v2/2016/01/"
filenames = [HKdataPath + "*_data_photon.root"]

for filename in filenames:
    # Get all files matching the pattern
    # Use glob to find all files that match the pattern
    files = sorted(glob.glob(filename))
print(files)
treename='HousekeepingData/SatStatus'

rdf_ = ROOT.RDataFrame(treename, files)
rdf_.GetColumnNames()
df_HK  = pd.DataFrame(rdf_.AsNumpy(['DmpHKDSatStatus.fSecond', 'DmpHKDSatStatus.fMicrosecond',
                    'DmpHKDSatStatus.fPosition.fX', 
                    'DmpHKDSatStatus.fPosition.fY', 
                    'DmpHKDSatStatus.fPosition.fZ',]))

In [ ]:
df_HK = df_HK.rename(columns={'DmpHKDSatStatus.fSecond' : 'second', 
                              'DmpHKDSatStatus.fMicrosecond' : 'microsecond',
                              'DmpHKDSatStatus.fPosition.fX' : 'X',
                               'DmpHKDSatStatus.fPosition.fY' : 'Y',
                                'DmpHKDSatStatus.fPosition.fZ' : 'Z' })
df_HK['millisecond'] = df_HK['microsecond'] * 1e-3
df_HK["MJD1"], df_HK["MJD2"] = met_to_mjd_tt_split(df_HK["second"], df_HK["millisecond"])
df_HK["MJD1"], df_HK["MJD2"] = add_leap_seconds(df_HK["MJD1"].values, df_HK["MJD2"].values)

In [ ]:
# Combine MJD1 + MJD2 into a single Time array
times = Time(val=[df_HK["MJD1"].values, df_HK["MJD2"].values], format="mjd", scale="tt")

print(times[:5])
# Compute absolute differences
time_diffs = np.abs(times - MJD_target)

# Find closest
closest_idx = time_diffs.argmin()
closest_row = df_HK.iloc[closest_idx]
delta_sec = (times[closest_idx] - MJD_target).sec
print(f"Time difference = {delta_sec:.3f} seconds")

